# Build KG for Beauty Data

1. Load raw data
+ reviews (all & 5core)
+ metadata

2. build item, user and charecteristic list
+ each enitity should have a unique id

3. Load train, test and validation data from huggingface


4. build KG
+ build triples from list



## Amazon Subsets

### 1. load raw data

In [1]:
import pandas as pd
import datasets

dataset = datasets.load_dataset("McAuley-Lab/Amazon-Reviews-2023", "5core_timestamp_Books")
metadata = datasets.load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_meta_Books")

# load 50% of the data
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
meta_df = pd.DataFrame(metadata["full"])

# drop duplicates 
meta_df = meta_df.drop_duplicates(subset=['parent_asin'])
metadata_filtered_df = meta_df[meta_df.parent_asin.isin(train_df.parent_asin)]


Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

In [15]:
# 20-core 
train_test_df = pd.concat([train_df, test_df], ignore_index=True)
unique_user_ids = train_test_df['user_id'].unique()

user_value_count = train_test_df['user_id'].value_counts()

# filter out users with less than 20 ratings
train_test_df = train_test_df[train_test_df['user_id'].isin(user_value_count[user_value_count >= 20].index)]

print("Filtered out % of users: ", (len(unique_user_ids) - len(train_test_df['user_id'].unique()))/len(unique_user_ids)*100, "%")

metadata_count = len(metadata_filtered_df.parent_asin.unique())

metadata_20core = metadata_filtered_df[metadata_filtered_df.parent_asin.isin(train_test_df.parent_asin)]

print("Filtered out % of metadata: ", (metadata_count - len(metadata_20core.parent_asin.unique()))/metadata_count*100, "%")
print("Number of users: ", len(train_test_df['user_id'].unique()))
print("Number of metadata: ", len(metadata_20core.parent_asin.unique()))


Filtered out % of users:  78.95630398336199 %
Filtered out % of metadata:  3.6646935933147633 %
Number of users:  19832
Number of metadata:  22134


In [19]:
from datasets import load_dataset
import re
import pandas as pd

class Preprocess_Amazon_Data:
    def __init__(self, dataset_name, metadataset_name, core_setting):
        self.dataset_name = dataset_name
        self.metadataset_name = metadataset_name
        self.core_setting = core_setting
    
    def load_data(self):

        metadata_df = load_dataset("McAuley-Lab/Amazon-Reviews-2023", self.metadataset_name)
        dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", self.dataset_name)

        train_df = pd.DataFrame(dataset["train"])
        test_df = pd.DataFrame(dataset["test"])
        meta_df = pd.DataFrame(metadata_df["full"])

        # drop duplicates 
        meta_df = meta_df.drop_duplicates(subset=['parent_asin'])
        
        train_test_df = pd.concat([train_df, test_df], ignore_index=True)
        user_value_count = train_test_df['user_id'].value_counts()
        item_value_count = train_test_df['parent_asin'].value_counts()
        train_test_df = train_test_df[train_test_df['user_id'].isin(user_value_count[user_value_count >= self.core_setting].index)]

        print("Filtered out % of users: ", (len(unique_user_ids) - len(train_test_df['user_id'].unique()))/len(unique_user_ids)*100, "%")
        print("Filtered out % of items: ", (len(item_value_count) - len(train_test_df['parent_asin'].unique()))/len(item_value_count)*100, "%")
        
        train_df = train_df[train_df.parent_asin.isin(train_test_df.parent_asin)]
        test_df = test_df[test_df.parent_asin.isin(train_test_df.parent_asin)]

        # filter metadata parent_asin which is in train_df or test_df
        metadata_filtered_df = meta_df[meta_df.parent_asin.isin(train_df.parent_asin) | meta_df.parent_asin.isin(test_df.parent_asin)]

        

        print("The meta dataset is reduced by ",(len(metadata_filtered_df)-len(meta_df))/len(meta_df)*100,"%", " reviews, because of 5core rating")

        return train_df, test_df, metadata_filtered_df
    
    def clean_string(self, string):
        string = re.sub(r'\[', '', string)
        string = re.sub(r'\]', '', string)
        string = re.sub(r'"', '', string)
        string = re.sub(r'\s+', ' ', string)
        string = re.sub("{", "", string)
        string = re.sub("}", "", string)
        return string
    
    def clean_column(self, df, column_name):
        def safe_clean(value):
            if isinstance(value, list):
                return self.clean_string(str(value))
            elif pd.isna(value) or value is None:
                return ""
            else:
                return self.clean_string(str(value))
        return df[column_name].apply(safe_clean)
    
    def create_source_text(self, product):
        """Concatenate product information into a text string

        Args:
            product (dict): dictionary containing product information

        Returns:
            str: concatenated product information
        """
        # TODO: update columns for other datasets
        description = "*" if product["description"] == "" else f"Description: {product['description']}"
        features = "*" if product["features"] == "" else f"Features: {product['features']}"
        details = "*" if product["details"] == "" else f"Details: {product['details']}"
        store = "*" if product["store"] == "" else f"Store: {product['store']}"
        categories = "*" if product["categories"] == "" else f"Categories: {product['categories']}"
        price = "*" if product["price"] == "" else f"Price: {product['price']}"
        author = "*" if product["author"] == "" else f"Author: {product['author']}"

        concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}; Author: {author}; Description: {description}; Features: {features} - {details}; Store: {store}; Categories: {categories}; Price: {price};"
        return concatenated_text
    
    def clean_text_columns(self,df, columns):
        """clean the text columns that might contain lists or dictionaries. 
        Concatenate the text columns into a single column called "source_text"
        Args:
            df (pd.DataFrame): The dataframe to clean
            columns (list): The list of columns to clean
        Returns:
            pd.DataFrame: The cleaned dataframe with the new "source_text" column
        """
        for column in columns:
            if column in df.columns:
                df[column] = self.clean_column(df, column)

        df['source_text'] = df.apply(self.create_source_text, axis=1)
        return df
    
if __name__ == "__main__":

    # select dataset to preprocess
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care", "Video_Games"]
    DATASET = DATASETS[0]
    CORE_SETTING = 10
    METADATA_NAME = "raw_meta_" + DATASET
    DATASET_NAME = "5core_timestamp_" + DATASET #keep 5core, because McAuley datasets only have 5core
    
    # select columns to preprocess
    COLUMNS = ['description', 'features', 'details', 'store', 'categories', 'price', 'author']

    # load data
    amazon_data_prep = Preprocess_Amazon_Data(DATASET_NAME, METADATA_NAME, CORE_SETTING)
    train_df, test_df, metadata_filtered_df = amazon_data_prep.load_data()

    # clean data
    metadata_filtered_df = amazon_data_prep.clean_text_columns(metadata_filtered_df, COLUMNS)

    # Create directory if it doesn't exist
    import os
    os.makedirs(f'../../data/preprocessed/amazon-{DATASET}', exist_ok=True)
    
    # Save data to the created directory
    metadata_filtered_df.to_csv(f'../../data/preprocessed/amazon-{DATASET}/metadata_filtered_df.csv', index=False)
    train_df.to_csv(f'../../data/preprocessed/amazon-{DATASET}/train_df.csv', index=False)
    test_df.to_csv(f'../../data/preprocessed/amazon-{DATASET}/test_df.csv', index=False)


Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

In [3]:
import re 
def create_source_text(product):
        """Concatenate product information into a text string

        Args:
            product (dict): dictionary containing product information

        Returns:
            str: concatenated product information
        """
        description = "*" if product["description"] == "" else f"Description: {product['description']}"
        features = "*" if product["features"] == "" else f"Features: {product['features']}"
        details = "*" if product["details"] == "" else f"Details: {product['details']}"
        store = "*" if product["store"] == "" else f"Store: {product['store']}"
        categories = "*" if product["categories"] == "" else f"Categories: {product['categories']}"
        price = "*" if product["price"] == "" else f"Price: {product['price']}"
        author = "*" if product["author"] == "" else f"Author: {product['author']}"

        # concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}; Author: {author}; Description: {description}; Features: {features} - {details}; Store: {store}; Categories: {categories}; Price: {price};"
        concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}Description: {description};- {details}; Store: {store};"
        #print("Concatenated text:", concatenated_text)  # Debug: Print the output text
        return concatenated_text
    
def clean_string(string):
            string = re.sub(r'\[', '', string)
            string = re.sub(r'\]', '', string)
            string = re.sub(r'"', '', string)
            string = re.sub(r'\s+', ' ', string)
            string = re.sub("{", "", string)
            string = re.sub("}", "", string)
            return string

def clean_column(df, column_name):
    """
    Clean a column in the dataframe by applying clean_string to each element.
    Handles lists by converting them to strings first.
    
    Args:
        df (pandas.DataFrame): The dataframe containing the column to clean
        column_name (str): The name of the column to clean
        
    Returns:
        pandas.Series: The cleaned column
    """
    def safe_clean(value):
        if isinstance(value, list):
            # Convert list to string before cleaning
            return clean_string(str(value))
        elif pd.isna(value) or value is None:
            return ""
        else:
            return clean_string(str(value))
    
    return df[column_name].apply(safe_clean)

def clean_text_columns(df, columns):
    """clean the text columns that might contain lists or dictionaries. 
    Concatenate the text columns into a single column called "source_text"
    Args:
        df (pd.DataFrame): The dataframe to clean
        columns (list): The list of columns to clean
    Returns:
        pd.DataFrame: The cleaned dataframe with the new "source_text" column
    """
    for column in columns:
        if column in df.columns:
            df[column] = clean_column(df, column)

    df['source_text'] = df.apply(create_source_text, axis=1)
    return df





/var/folders/33/fw5wlwpj4b78_xvtgsm72q6r0000gq/T/ipykernel_48037/1394984172.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered_df[column] = clean_column(metadata_filtered_df, column)
/var/folders/33/fw5wlwpj4b78_xvtgsm72q6r0000gq/T/ipykernel_48037/1394984172.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  metadata_filtered_df['source_text'] = metadata_filtered_df.apply(create_source_text, axis=1)


### 2. build item, user and charecteristic list


#### item user list 


## Million Song Dataset

+ load data
+ transform data into a dataframe: _split into msd_track_id, musicxmatch_track_id, word_indices_ 

Links for train and test data:
[The musiXmatch Dataset](http://millionsongdataset.com/musixmatch/)



In [110]:
import pandas as pd 



# If the data is comma-separated within each row, we can split it
# Based on the output, it seems each row has a format like:
# TRZZZWS128F429CF87,3080645,6:1,24:9,38:7,42:1,...
# Let's parse this into separate columns
# The error suggests there's an issue with the number of fields in the data
# Let's try a different approach by reading the file as a text file first
# and then parsing it manually

# Reset the file reading approach
with open('../../data/raw/Last-FM/mxm_lyrics_train.txt', 'r') as f:
    lines = f.readlines()

# Skip comment lines
data_lines = [line.strip() for line in lines if not line.startswith('#')]

# The first line might be a header with column names
header = data_lines[0].split(',')
data = data_lines[1:]

# Create a DataFrame with a single column containing the entire line
lastfm_lyrics_df = pd.DataFrame(data, columns=[0])

# If we need to parse the comma-separated values within each row later,
# we can do that with a custom function based on the data structure

# First, let's examine the structure of the first few rows to understand the format
print("First row structure:")
print(lastfm_lyrics_df.iloc[0, 0].split(',')[:5])  # Print first 5 elements after splitting

# Based on the first row structure, it seems each row has:
# 1. A track ID (e.g., TRAAAAV128F421A322)
# 2. A number (e.g., 4623710)
# 3. Word counts in format word_id:count (e.g., 1:6,2:4,3:2,...)

# Let's split each row into track_id and word_counts
def parse_lastfm_row(row):
    # Split on the first comma to separate track_id_
    parts = row.split(',', 1)
    msd_track_id = parts[0]

    # split on second comma to seperate musicxmatch word for second comma
    parts_word_counts = parts[1].split(',', 1)
    musicxmatch_track_id = parts_word_counts[0]
    # get from every word_index_count tuple only the word_index
    # Extract all word indices from the word_counts part (which is parts_word_counts[1])
    # Format is like "1:6,2:4,3:2,..." so we split by comma and then by colon to get indices
    word_counts_str = parts_word_counts[1]
    word_indices = [item.split(':')[0] for item in word_counts_str.split(',')]
    

    return msd_track_id, musicxmatch_track_id, word_indices

# Apply the parsing function to create new columns
lastfm_lyrics_df[['msd_track_id', 'musicxmatch_track_id', 'word_indices']] = lastfm_lyrics_df[0].apply(lambda x: pd.Series(parse_lastfm_row(x)))

# Drop the original column
lastfm_lyrics_df = lastfm_lyrics_df.drop(columns=[0])

# load test df 
# Reset the file reading approach
with open('../../data/raw/Last-FM/mxm_lyrics_test.txt', 'r') as f:
    lines = f.readlines()

# Skip comment lines
data_lines = [line.strip() for line in lines if not line.startswith('#')]

# The first line might be a header with column names
header = data_lines[0].split(',')
data = data_lines[1:]

# Create a DataFrame with a single column containing the entire line
lastfm_test_df = pd.DataFrame(data, columns=[0])

# apply parsing function to test df
lastfm_test_df[['msd_track_id', 'musicxmatch_track_id', 'word_indices']] = lastfm_test_df[0].apply(lambda x: pd.Series(parse_lastfm_row(x)))

# drop original column
lastfm_test_df = lastfm_test_df.drop(columns=[0])

# append  train and test df
lastfm_lyrics_df = pd.concat([lastfm_lyrics_df, lastfm_test_df], ignore_index=True)

First row structure:
['TRAAAAV128F421A322', '4623710', '1:6', '2:4', '3:2']


load mappings from word index to word

In [111]:
word_index_to_word = {}

In [112]:
# load train and test df
# Reset the file reading approach
with open('../../data/raw/Last-FM/mxm_lyrics_test.txt', 'r') as f:
    lines = f.readlines()

# only include line starting with %, contains word list 
data_lines = [line.strip() for line in lines if line.startswith('%')]
words = data_lines[0].split(',') # split on seperator
# remove % from first element
words[0] = words[0].replace('%', '')

# create a dictionary with a single column containing the entire line
word_to_index_test = dict(zip(range(1, len(words) + 1), words))


In [113]:
# load train df
with open('../../data/raw/Last-FM/mxm_lyrics_train.txt', 'r') as f:
    lines = f.readlines()

# only include line starting with %, contains word list 
data_lines = [line.strip() for line in lines if line.startswith('%')]
words = data_lines[0].split(',') # split on seperator


# create a dictionary with a single column containing the entire line
word_to_index_train = dict(zip(range(1, len(words) + 1), words))

In [114]:
# merge train and test dict
word_index_to_word = {**word_to_index_train, **word_to_index_test}

# drop duplicates
word_index_to_word = dict(sorted(word_index_to_word.items(), key=lambda item: item[0]))


map word_indices to word_index_to_word


In [115]:
lastfm_lyrics_df['lyrics'] = lastfm_lyrics_df['word_indices'].apply(lambda x: [word_index_to_word[int(index)] for index in x])



,msd_track_id,musicxmatch_track_id,word_indices,lyrics
0,TRAAAAV128F421A322,4623710,"[1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 1...","[i, the, you, to, and, a, me, it, my, is, of, ..."
1,TRAAABD128F429CF47,6477168,"[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 1...","[i, you, to, and, a, me, it, not, in, my, is, ..."
2,TRAAAED128E0783FAB,2516445,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15...","[i, the, you, to, and, a, me, it, not, in, my,..."
3,TRAAAEF128F4273421,3759847,"[1, 2, 3, 4, 5, 6, 9, 12, 13, 15, 16, 18, 23, ...","[i, the, you, to, and, a, not, is, of, that, d..."
4,TRAAAEW128F42930C0,3783760,"[1, 4, 5, 6, 7, 9, 10, 11, 15, 17, 20, 22, 32,...","[i, to, and, a, me, not, in, my, that, on, am,..."


convert lyrics to string

In [116]:
#convert lyrics to string by removing commas
# Convert lyrics to string by joining the words and removing commas
lastfm_lyrics_df['lyrics'] = lastfm_lyrics_df['lyrics'].apply(lambda x: ' '.join([item.replace(',', '') for item in x]))

,msd_track_id,musicxmatch_track_id,word_indices,lyrics
0,TRAAAAV128F421A322,4623710,"[1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 1...",i the you to and a me it my is of your that ar...
1,TRAAABD128F429CF47,6477168,"[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 1...",i you to and a me it not in my is your that do...
2,TRAAAED128E0783FAB,2516445,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15...",i the you to and a me it not in my is of that ...
3,TRAAAEF128F4273421,3759847,"[1, 2, 3, 4, 5, 6, 9, 12, 13, 15, 16, 18, 23, ...",i the you to and a not is of that do are for k...
4,TRAAAEW128F42930C0,3783760,"[1, 4, 5, 6, 7, 9, 10, 11, 15, 17, 20, 22, 32,...",i to and a me not in my that on am all with li...


remove stopwords

+ Helps reduce noise of words for experiments.
    + More attention on words that are more relevant for the experiment (NER, Keywords, or LLM) 


merge metadata to lastfm_df

In [117]:
with open("../../data/raw/Last-FM/mxm_metadata.txt", "r") as f:
    lines = f.readlines()

# skip lines starting with #
lines = [line.strip() for line in lines if not line.startswith('#')]

metadata_df = pd.DataFrame(lines, columns=["metadata"])

In [118]:
# Parse the metadata into columns
# The format is: tid<SEP>artist name<SEP>title<SEP>mxm tid<SEP>artist_name<SEP>title
# Where:
# tid -> Million Song Dataset track ID
# artist name -> artist name in the MSD
# title -> title in the MSD
# mxm tid -> musiXmatch track ID
# artist name -> artist name for mXm
# title -> title for mXm

# Split the metadata into columns
metadata_df[['msd_track_id', 'artist_name', 'title', 'mxm_track_id', 'artist_name_mxm', 'title_mxm']] = metadata_df['metadata'].str.split('<SEP>', expand=True)

# Drop the original metadata column
metadata_df = metadata_df.drop(columns=['metadata'])



,msd_track_id,artist_name,title,mxm_track_id,artist_name_mxm,title_mxm
0,TRMMMKD128F425225D,Karkkiautomaatti,Tanssi vaan,4418550,Karkkiautomaatti,Tanssi vaan
1,TRMMMRX128F93187D9,Hudson Mohawke,No One Could Ever,8898149,Hudson Mohawke,No One Could Ever
2,TRMMMCH128F425532C,Yerba Brava,Si Vos Querés,9239868,Yerba Brava,Si vos queres
3,TRMMMXN128F42936A5,David Montgomery,"Symphony No. 1 G minor ""Sinfonie Serieuse""/All...",5346741,Franz Berwald,"Symphony No. 1 in G minor ""Sinfonie Sérieuse"":..."
4,TRMMMBB12903CB7D21,Kris Kross,2 Da Beat Ch'yall,2511405,Kris Kross,2 Da Beat Ch'yall


merge metadf and lastfm_lyrics_df

In [121]:

lastfm_metadata_df = pd.merge(lastfm_lyrics_df, metadata_df, on='msd_track_id', how='left',)

# filter out columns that are not needed
lastfm_metadata_df = lastfm_metadata_df.drop(columns=['mxm_track_id', 'musicxmatch_track_id', 'title_mxm'])


,msd_track_id,word_indices,lyrics,artist_name,title,artist_name_mxm
0,TRAAAAV128F421A322,"[1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15, 1...",i the you to and a me it my is of your that ar...,Western Addiction,A Poor Recipe For Civic Cohesion,Western Addiction
1,TRAAABD128F429CF47,"[1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 1...",i you to and a me it not in my is your that do...,The Box Tops,Soul Deep,The Box Tops
2,TRAAAED128E0783FAB,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15...",i the you to and a me it not in my is of that ...,Jamie Cullum,It's About Time,Jamie Cullum
3,TRAAAEF128F4273421,"[1, 2, 3, 4, 5, 6, 9, 12, 13, 15, 16, 18, 23, ...",i the you to and a not is of that do are for k...,Adam Ant,Something Girls,Adam Ant
4,TRAAAEW128F42930C0,"[1, 4, 5, 6, 7, 9, 10, 11, 15, 17, 20, 22, 32,...",i to and a me not in my that on am all with li...,Broken Spindles,Burn My Body (Album Version),Broken Spindles


load user listening history 

+ check if track_id matches with msd_track_id
+ prepare for 10 core rating 


[Million Song Dataset User Listening History](https://www.kaggle.com/datasets/undefinenull/million-song-dataset-spotify-lastfm?resource=download&select=User+Listening+History.csv)


In [127]:
user_listening_history = pd.read_csv("../../data/raw/Last-FM/msd_user_listening_history.csv")
# rename columns
user_listening_history = user_listening_history.rename(columns={'track_id': 'msd_track_id'})

rows = len(user_listening_history)

# filter out rows where msd_track_id is not in lastfm_metadata_df
user_listening_history = user_listening_history[user_listening_history['msd_track_id'].isin(lastfm_metadata_df['msd_track_id'])]

print("Filtered out % ", (rows - len(user_listening_history))/rows*100, " rows")


Filtered out %  33.122348900523214  rows


filter out users with less than 30 ratings (30-Core)

In comparisopn to the amazon dataset, where we use 5-core, we use 30-core here. Reason is that the lastfm dataset is much larger than the amazon dataset. 

In [131]:
# Count the number of ratings per user
user_counts = user_listening_history['user_id'].value_counts()

# Get users with at least 5 ratings
users_to_keep = user_counts[user_counts >= 30].index

# Filter the dataframe to only include these users
user_listening_history_5core = user_listening_history[user_listening_history['user_id'].isin(users_to_keep)]


# Print statistics
print(f"Original number of users: {user_listening_history['user_id'].nunique()}")
print(f"Number of users after 5-core filtering: {user_listening_history_5core['user_id'].nunique()}")
print(f"Original number of records: {len(user_listening_history)}")
print(f"Number of records after 5-core filtering: {len(user_listening_history_5core)}")
print(f"Percentage of records kept: {len(user_listening_history_5core)/len(user_listening_history)*100:.2f}%")

# Replace the original dataframe with the filtered one
user_listening_history = user_listening_history_5core


Original number of users: 71833
Number of users after 5-core filtering: 33534
Original number of records: 2502125
Number of records after 5-core filtering: 1593130
Percentage of records kept: 63.67%


split into train and test df

In [133]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(user_listening_history, test_size=0.2, random_state=42)


save data

In [135]:
train_df.to_csv("../../data/preprocessed/Last-FM/train_df.csv", index=False)
test_df.to_csv("../../data/preprocessed/Last-FM/test_df.csv", index=False)
metadata_df.to_csv("../../data/preprocessed/Last-FM/metadata_df_filtered.csv", index=False)